In [1]:

print("Ok")


Ok


In [2]:

%pwd

'c:\\projects\\medibot\\Elarova\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\projects\\medibot\\Elarova'

In [5]:

from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [ ]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [ ]:
extracted_data=load_pdf_file(data='Data/')

In [ ]:
# extracted_data

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [ ]:
# text_chunks

In [ ]:

from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:

#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:

embeddings = download_hugging_face_embeddings()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14296\2503581564.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


In [ ]:

query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [ ]:
# query_result

In [ ]:

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GEMINI_API_KEY=os.environ.get('GEMINI_API_KEY')

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

# Retrieve Pinecone API key (replace "PINECONE_API_KEY" with your actual API key)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY environment variable is not set.")

# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Define index parameters
index_name = "elarova"
dimension = 384
metric = "cosine"
cloud = "aws"
region = "us-east-1"

# Check if the index already exists
if index_name in pc.list_indexes():
    print(f"Index '{index_name}' already exists.")
else:
    try:
        # Create the index
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(
                cloud=cloud,
                region=region
            )
        )
        print(f"Index '{index_name}' created successfully.")
    except Exception as e:
        print(f"Failed to create index: {e}")

Index 'elarova' created successfully.


In [ ]:
import os

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [ ]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [ ]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acoustic neuroma?")

In [ ]:

retrieved_docs

[Document(id='54f32c90-b302-4233-9f2e-34eb38d19b79', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 42.0, 'page_label': '43', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 29\nAcoustic neuroma\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 29'),
 Document(id='44bfac14-fe6d-4145-a6ed-4091be0c1109', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 40.0, 'page_label': '41', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='Acoustic neuroma\nDefinition\nAn acoustic neuroma is a benign tumor involving\ncells of the myelin sheath that surrounds the vestibulo-\ncochlear nerve (eighth cranial nerve).\nDescription\nThe vestibulocochlear nerve extends from the inner\near to the brain and 

In [ ]:
pip install langchain


Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip show pydantic langchain langchain_openai langchain-core


Name: pydantic
Version: 2.11.5
Summary: Data validation using Python type hints
Home-page: https://github.com/pydantic/pydantic
Author: 
Author-email: Samuel Colvin <s@muelcolvin.com>, Eric Jolibois <em.jolibois@gmail.com>, Hasan Ramezani <hasan.r67@gmail.com>, Adrian Garcia Badaracco <1755071+adriangb@users.noreply.github.com>, Terrence Dorsey <terry@pydantic.dev>, David Montague <david@pydantic.dev>, Serge Matveenko <lig@countzero.co>, Marcelo Trylesinski <marcelotryle@gmail.com>, Sydney Runkle <sydneymarierunkle@gmail.com>, David Hewitt <mail@davidhewitt.io>, Alex Hall <alex.mojaki@gmail.com>, Victorien Plot <contact@vctrn.dev>
License-Expression: MIT
Location: e:\anaconda\envs\elarova\lib\site-packages
Requires: annotated-types, pydantic-core, typing-extensions, typing-inspection
Required-by: langchain, langchain-core, langsmith, openai, pydantic-settings
---
Name: langchain
Version: 0.3.25
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-e

In [ ]:
pip install --upgrade pydantic langchain langchain-openai langchain-core


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b", temperature=0.4)


In [ ]:
pip install langchain


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are Elarova, a professional medical assistant chatbot. "
    "Answer questions using the retrieved context below. Provide clear, concise responses "
    "limited to three sentences. Include relevant examples when helpful. "
    "If uncertain or the answer is not in the context, say you don't know. "
    "Always highlight or reference the source of your information from the context."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}